In [39]:
import numpy as np
import tensorflow as tf
import wave,math,struct
from keras.models import Sequential
from keras.layers import Dense, LSTM, Activation


## Prepare a dummy data for simulating notes

In [40]:
notes_freq = {'A': 440.0, 'B':493.88, 'C': 261.63, 'D': 293.66,
              'E' : 393.63, 'F' : 349.23, 'G': 392.00
               }
notes_freq

{'A': 440.0,
 'B': 493.88,
 'C': 261.63,
 'D': 293.66,
 'E': 393.63,
 'F': 349.23,
 'G': 392.0}

In [41]:
notes = list(notes_freq.keys())
notes

['A', 'B', 'C', 'D', 'E', 'F', 'G']

In [42]:
note_to_int = {note:i for i,note in enumerate(notes)}
note_to_int

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}

In [43]:
int_to_note = {i:note for i,note in enumerate(notes)}
int_to_note

{0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G'}

In [44]:
raw_music_data = [notes[np.random.randint(0,7)] for i in range (1000)]
raw_music_data

['A',
 'F',
 'D',
 'E',
 'D',
 'B',
 'E',
 'D',
 'G',
 'B',
 'G',
 'C',
 'E',
 'F',
 'D',
 'F',
 'C',
 'E',
 'E',
 'C',
 'B',
 'D',
 'E',
 'C',
 'C',
 'G',
 'B',
 'G',
 'B',
 'G',
 'A',
 'G',
 'D',
 'C',
 'F',
 'G',
 'B',
 'B',
 'G',
 'A',
 'C',
 'C',
 'B',
 'C',
 'F',
 'E',
 'C',
 'C',
 'D',
 'F',
 'C',
 'F',
 'E',
 'G',
 'C',
 'F',
 'F',
 'E',
 'C',
 'E',
 'C',
 'E',
 'F',
 'B',
 'D',
 'B',
 'F',
 'D',
 'B',
 'A',
 'G',
 'G',
 'D',
 'A',
 'C',
 'C',
 'D',
 'A',
 'G',
 'B',
 'F',
 'G',
 'D',
 'C',
 'A',
 'D',
 'G',
 'D',
 'D',
 'G',
 'G',
 'A',
 'F',
 'C',
 'A',
 'C',
 'D',
 'B',
 'A',
 'E',
 'E',
 'F',
 'A',
 'C',
 'D',
 'E',
 'F',
 'C',
 'B',
 'A',
 'F',
 'F',
 'B',
 'A',
 'A',
 'F',
 'B',
 'D',
 'C',
 'G',
 'C',
 'D',
 'G',
 'G',
 'F',
 'B',
 'F',
 'B',
 'D',
 'G',
 'B',
 'A',
 'F',
 'G',
 'C',
 'B',
 'B',
 'C',
 'D',
 'D',
 'G',
 'A',
 'D',
 'E',
 'F',
 'A',
 'C',
 'B',
 'G',
 'G',
 'B',
 'G',
 'E',
 'G',
 'B',
 'F',
 'B',
 'F',
 'C',
 'D',
 'C',
 'B',
 'A',
 'D',
 'G',
 'B',
 'C'

## Data Preparation


In [45]:
sequence_length = 3
network_input = []
network_output = []

for i in range (len(raw_music_data) - sequence_length):
    sequence_in = raw_music_data[i:i+sequence_length]
    sequence_out = raw_music_data[i+sequence_length]
    network_input.append([note_to_int[char] for char in sequence_in])
    network_output.append(note_to_int[sequence_out])
    print(sequence_in,'--->',sequence_out)

['A', 'F', 'D'] ---> E
['F', 'D', 'E'] ---> D
['D', 'E', 'D'] ---> B
['E', 'D', 'B'] ---> E
['D', 'B', 'E'] ---> D
['B', 'E', 'D'] ---> G
['E', 'D', 'G'] ---> B
['D', 'G', 'B'] ---> G
['G', 'B', 'G'] ---> C
['B', 'G', 'C'] ---> E
['G', 'C', 'E'] ---> F
['C', 'E', 'F'] ---> D
['E', 'F', 'D'] ---> F
['F', 'D', 'F'] ---> C
['D', 'F', 'C'] ---> E
['F', 'C', 'E'] ---> E
['C', 'E', 'E'] ---> C
['E', 'E', 'C'] ---> B
['E', 'C', 'B'] ---> D
['C', 'B', 'D'] ---> E
['B', 'D', 'E'] ---> C
['D', 'E', 'C'] ---> C
['E', 'C', 'C'] ---> G
['C', 'C', 'G'] ---> B
['C', 'G', 'B'] ---> G
['G', 'B', 'G'] ---> B
['B', 'G', 'B'] ---> G
['G', 'B', 'G'] ---> A
['B', 'G', 'A'] ---> G
['G', 'A', 'G'] ---> D
['A', 'G', 'D'] ---> C
['G', 'D', 'C'] ---> F
['D', 'C', 'F'] ---> G
['C', 'F', 'G'] ---> B
['F', 'G', 'B'] ---> B
['G', 'B', 'B'] ---> G
['B', 'B', 'G'] ---> A
['B', 'G', 'A'] ---> C
['G', 'A', 'C'] ---> C
['A', 'C', 'C'] ---> B
['C', 'C', 'B'] ---> C
['C', 'B', 'C'] ---> F
['B', 'C', 'F'] ---> E
['C', 'F', 

In [46]:
network_input


[[0, 5, 3],
 [5, 3, 4],
 [3, 4, 3],
 [4, 3, 1],
 [3, 1, 4],
 [1, 4, 3],
 [4, 3, 6],
 [3, 6, 1],
 [6, 1, 6],
 [1, 6, 2],
 [6, 2, 4],
 [2, 4, 5],
 [4, 5, 3],
 [5, 3, 5],
 [3, 5, 2],
 [5, 2, 4],
 [2, 4, 4],
 [4, 4, 2],
 [4, 2, 1],
 [2, 1, 3],
 [1, 3, 4],
 [3, 4, 2],
 [4, 2, 2],
 [2, 2, 6],
 [2, 6, 1],
 [6, 1, 6],
 [1, 6, 1],
 [6, 1, 6],
 [1, 6, 0],
 [6, 0, 6],
 [0, 6, 3],
 [6, 3, 2],
 [3, 2, 5],
 [2, 5, 6],
 [5, 6, 1],
 [6, 1, 1],
 [1, 1, 6],
 [1, 6, 0],
 [6, 0, 2],
 [0, 2, 2],
 [2, 2, 1],
 [2, 1, 2],
 [1, 2, 5],
 [2, 5, 4],
 [5, 4, 2],
 [4, 2, 2],
 [2, 2, 3],
 [2, 3, 5],
 [3, 5, 2],
 [5, 2, 5],
 [2, 5, 4],
 [5, 4, 6],
 [4, 6, 2],
 [6, 2, 5],
 [2, 5, 5],
 [5, 5, 4],
 [5, 4, 2],
 [4, 2, 4],
 [2, 4, 2],
 [4, 2, 4],
 [2, 4, 5],
 [4, 5, 1],
 [5, 1, 3],
 [1, 3, 1],
 [3, 1, 5],
 [1, 5, 3],
 [5, 3, 1],
 [3, 1, 0],
 [1, 0, 6],
 [0, 6, 6],
 [6, 6, 3],
 [6, 3, 0],
 [3, 0, 2],
 [0, 2, 2],
 [2, 2, 3],
 [2, 3, 0],
 [3, 0, 6],
 [0, 6, 1],
 [6, 1, 5],
 [1, 5, 6],
 [5, 6, 3],
 [6, 3, 2],
 [3, 2, 0],
 [2,

In [47]:
n_patterns = len(network_input)
n_patterns


997

In [48]:
x=np.reshape(network_input,(n_patterns,sequence_length,1))#syntax = samples,timestamps,features

In [49]:
x.shape

(997, 3, 1)

In [50]:
from keras.utils import to_categorical

In [51]:
y = to_categorical(network_output)
y.shape

(997, 7)

In [52]:
y

array([[0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(997, 7))

## Build the model

In [53]:
model = Sequential()
model.add(LSTM(256, input_shape=(sequence_length,1)))
model.add(Dense(1000,activation='relu'))
model.add(Dense(7,activation='softmax'))
model.compile(loss='categorical_crossentropy',optimizer='adam')

## Train the model

In [54]:
model.fit(x,y, epochs= 100)

Epoch 1/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.9567
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.9452 
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.9467 
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.9446 
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.9445 
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.9420 
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.9420 
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.9390 
Epoch 9/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.9383 
Epoch 10/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.9403 
Epoch 11/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.9371 
Epoch 12/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.9375 
Epoch 13/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.9365 
Epoch 14/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.9354 
Epoch 15/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss

### Generate a New Melody Sequence

In [55]:
start_index = np.random.randint(0,len(network_input))
pattern = network_input[start_index]
pattern

[0, 6, 5]

## Predict

In [56]:
generated_melody = []
for i in range (16):
    x_input = np.reshape(pattern,(1,len(pattern),1))
    prediction = model.predict(x_input,verbose=0)
    index = np.argmax(prediction)
    result = int_to_note[index]
    generated_melody.append(result)
    pattern.append(index)
    pattern=pattern[1:len(pattern)]

In [57]:
pattern

[np.int64(0), np.int64(2), np.int64(4)]

In [58]:
generated_melody

['C',
 'G',
 'B',
 'G',
 'G',
 'B',
 'A',
 'E',
 'E',
 'C',
 'D',
 'D',
 'G',
 'A',
 'C',
 'E']

## Save this a audio file

In [59]:
import wave, math, struct # Ensure wave and struct are imported

with wave.open('my_music.wav', 'w') as wave_file:
  # channel(mono),byte size, sampling rate
  wave_file.setparams((1,2,44100, 0, 'NONE','not compressed'))
  for note in generated_melody:
    freq = float(notes_freq[note]) # Convert freq to float
    num_samples = int(0.5 * 44100) # duration * sample rate
    for i in range(num_samples):
      #sample_rate
      t = float(i) / 44100 # t should be based on the sample index, not always 1
      value = int (32767 * 0.5 * math.sin(2 * math.pi * freq * t))
      data = struct.pack('<h', value)
      wave_file.writeframes(data)

In [60]:
help(wave.open)

Help on function open in module wave:

open(f, mode=None)

